# RunPod: A4, A3 with stop-gradient on the prediction target

**An ablation, not a model candidate.** A3 (B1's model with an MLP encoder on the simulator state) with the prediction target detached (`level1.wm.detach_pred_target=true`), the state twin of V3. The question: does stopping the gradient through the target path keep what A3's encoder removes, and does the latent still hold the scene, the neighbouring tiles and the mobs when the predictor cannot pull on the target? The reference run A3 comes from the Hub (`race/` only). No probe suite (it reads pixels): the readout of the race is the output, and the loss curves.

The race of the first epochs (README, "Ids"): the hypothesis is that the encoder removes, from the first steps, the variance the predictor cannot predict yet, before the predictor learns it. The model is saved at the untrained start and at many steps of the first two epochs (`dense_checkpoints`: steps 50, 100, ..., 2000, then every 1000), and after the training `race.evaluate` reads the latent of every checkpoint on a fixed sample of validation episodes (`slowness`'s nested sample): the linear read of the inventory, vitals, daylight, mobs and tiles in view, the four neighbouring tiles, the effective rank and the dead axes, the share of the variance inside a clip, and the predictor's error over copying. The four runs of the race are one 2x2: the encoder reads the pixels (B2 = B1 again, V3 = with stop-gradient) or the simulator state (A3, A4 = the same with an MLP encoder, no ViT, no CLS token), and the prediction target carries the gradient (B2, A3) or is detached (V3, A4). They train in parallel on their own pods; the last step of every notebook takes the others' `race/` files from the Hub and writes the report of all that exist.

Once per pod, in a terminal: `git clone https://github.com/quastAI/AAC.git /root/AAC`, then open this notebook from there. The paths and the helpers are in `notebooks/pod.py`, shared with the other pod notebooks.

| step | what |
|---|---|
| 0 | configuration, Hugging Face login, the PyTorch environment, the dataset from the Hub |
| 1 | a **smoke run** of this run on the real files (20 batches, a few dense checkpoints, then `race.evaluate` on 16 episodes) |
| 2 | `experiments/run.py run A4`, detached: training (dense checkpoints in the first two epochs; resumes after a pod restart), `race.evaluate`, `reports/A4_trace.md` |
| 3 | the four runs side by side (`reports/A4_race_all.md`: every run whose `race/` is on the Hub or here), then this run and its reports to the Hub (the dense checkpoints stay on the pod) |

A step that finished is skipped when you run its cell again.

In [ ]:
# --- 0. Configuration ------------------------------------------------------------------------------------------------
import os, shutil, sys, time
from pathlib import Path

EXPERIMENT = "A4"                          # the id: the experiment and its one trained run in experiments/experiments.yaml (README, "Ids")
COMPARE = ["B2", "V3", "A3", "A4"]          # the four runs of the race (pixels or state x gradient through the target or stop-gradient)

REPO = Path("/root/AAC")                    # the git clone, on the local disk (pod.py has the other paths and why they are where they are)
sys.path.insert(0, str(REPO / "notebooks"))
import pod

HF_REPO_ID = "quastAI/craftax-classic-pqn-rnn-v1"   # the dataset repo of runpod_dataset.ipynb (data/), runs go to runs/
HF_PRIVATE = True
SEEDS = "42"                                # one seed per run
MAX_EPOCHS = 10                             # as B1: the cosine schedule ends here (B2 stops after epoch 2 with this schedule)
USE_WANDB = True                            # live curves in Weights & Biases (project hjepa-flat); False = the metrics CSV only

In [ ]:
# --- 0. Paths, Hugging Face login, the PyTorch environment (installed again on a new pod) and the dataset ----------------
pod.configure(repo=REPO, seeds=SEEDS)
from pod import DATA, H_JEPA, PYTHON, RUNNER, launch, progress, run_dir, running, sh, show, wait

TRAIN_OVERRIDES = pod.train_overrides(USE_WANDB)   # Hydra overrides for every training run
api = pod.login(HF_REPO_ID, HF_PRIVATE, USE_WANDB)
pod.install(api, HF_REPO_ID, USE_WANDB)
SPEC = pod.run_spec(EXPERIMENT, EXPERIMENT)        # the run's training config and overrides (its run is named by the id)
FOLDER = pod.folder_of(EXPERIMENT)         # the checkpoint folder: B1 and A1 keep the ones they were trained in, the others train into their id
print(EXPERIMENT, FOLDER, SPEC["config"], *SPEC.get("overrides", []))

## 1. A smoke run

The smoke run trains this run for 20 batches with its own overrides and a few dense checkpoints (steps 5 and 10, then every 10), then reads their latents on 16 validation episodes: a wrong override or a crash in the training, in `DenseCheckpoints` or in `race.evaluate` shows here, not after hours.

In [ ]:
# --- 1. What this pod's volume has is kept; a resumed training continues from lightning_resume/last.ckpt -----------------------------------
print(EXPERIMENT, "folder", FOLDER, "- resumable:", (run_dir(FOLDER) / "lightning_resume" / "last.ckpt").exists())

In [ ]:
# --- 1. Smoke run of this run on the real files, then the race readout on 16 episodes (the full output is in LOGS/smoke.log) --------
SMOKE = f"{EXPERIMENT}_smoke"
VAL = DATA / "craftax_expert_val.h5"
shutil.rmtree(run_dir(SMOKE), ignore_errors=True)
t0 = time.time()
sh([PYTHON, "main_hjepa.py", "--config-name", SPEC["config"], *SPEC.get("overrides", []), f"output_model_name={SMOKE}",
    f"seed={SEEDS.split(',')[0]}", "trainer.max_epochs=1", "+trainer.limit_train_batches=20", "+trainer.limit_val_batches=2",
    "dense_checkpoints.steps=[5,10]", "dense_checkpoints.every=10", "dense_checkpoints.stop_after_epochs=null",
    *TRAIN_OVERRIDES, "wandb.enabled=false"], cwd=H_JEPA, quiet=True, log="smoke")   # no W&B run for a smoke run
out = run_dir(SMOKE)
assert (out / f"{SMOKE}_object.ckpt").exists() and (out / f"{SMOKE}_step_0_object.ckpt").exists(), "the smoke run wrote no model or no dense checkpoint"
sh([PYTHON, "-m", "race.evaluate", "--run", out, "--val-file", VAL, "--max-val-episodes", "16"], cwd=H_JEPA, quiet=True, log="smoke_race")
assert (out / "race" / "trace.json").exists(), "race.evaluate wrote no trace"
print(f"smoke run ok in {time.time() - t0:.0f} s: {out}")

## 2. Training, the race readout

One detached job chain (`run.py run EXPERIMENT`): the training with dense checkpoints (B2 stops after epoch 2 with the 10-epoch schedule; the others train 10 epochs, about 3.3 hours on an A100 for the pixel runs), `race.evaluate` on every checkpoint (`<run>/race/trace.json`, resumable) and `race.report` (`reports/EXPERIMENT_trace.md`). If the pod restarts, run the start cell again: the training resumes from `lightning_resume/last.ckpt`, the finished jobs are skipped. The dense checkpoints are written to the network volume (about 30 files in two epochs, not often), the logs and stable-pretraining's folder stay local.

In [ ]:
# --- 2. Start (or resume) the run, detached ----------------------------------------------------------------------------------
OVERRIDES = [f"trainer.max_epochs={MAX_EPOCHS}", *TRAIN_OVERRIDES]
sh([*RUNNER, "plan", EXPERIMENT, "--seeds", SEEDS, "--override", *OVERRIDES])
launch(EXPERIMENT, [*RUNNER, "run", EXPERIMENT, "--mode", "local", "--seeds", SEEDS, "--override", *OVERRIDES])

In [ ]:
# --- 2. Monitor (run as often as you like) -----------------------------------------------------------------------------------------
print(EXPERIMENT, "RUNNING" if running(EXPERIMENT) else "not running")
print("\n".join(progress(EXPERIMENT, "run  ", 1)))   # the job that runs now
print("\n".join(progress(EXPERIMENT, "Epoch", 2) + progress(EXPERIMENT, "[health]", 3) + progress(EXPERIMENT, "[race]", 2)))
for p in sorted((run_dir(FOLDER) / "health").glob("epoch_*.json"))[-1:]:
    print("last epoch report:", p.name)

In [ ]:
# --- 2. Loss curves (run as often as you like, also while it trains: the metrics CSV is written as it goes) ----------------------
from IPython.display import Image, display
sh([PYTHON, REPO / "analysis" / "plot_training.py", "--run", run_dir(FOLDER)])
display(Image(filename=str(run_dir(FOLDER) / "health" / "loss_curves.png")))

In [ ]:
# --- 2. Wait for the chain, then the reports: training, the race of this run ------------------------------------------------------------
wait(EXPERIMENT, "Epoch")
show(run_dir(FOLDER) / "health" / "report.md", 40)
for report in sorted((DATA / "reports").glob(f"{EXPERIMENT}_*.md")):
    print("\n=====", report.name)
    show(report, 80)

## 3. The four runs, upload

The `race/` files of the other runs come from the Hub (only they, not the models); the report covers every run of `COMPARE` that has them. This run (its `race/`, its model and reports; the dense and epoch checkpoints and the resumable state are left out) and the reports go to `runs/` and `reports/` of the Hub repo.

In [ ]:
# --- 3. The race of all runs that exist, then this run and its reports to the Hub ---------------------------------------------------------
from IPython.display import Image, display
runs = {}
for model in COMPARE:
    folder = pod.folder_of(model)
    if model != EXPERIMENT:
        pod.fetch(api, HF_REPO_ID, f"{folder}/seed{SEEDS.split(',')[0]}", required=False, only="race/")
    if (run_dir(folder) / "race" / "trace.json").exists():
        runs[model] = run_dir(folder)
print("race of:", ", ".join(runs), "- missing:", ", ".join(m for m in COMPARE if m not in runs) or "none")
report = DATA / "reports" / f"{EXPERIMENT}_race_all.md"
sh([PYTHON, "-m", "race.report", "--out", report, *[f"{m}={d}" for m, d in runs.items()]], cwd=H_JEPA)
show(report, 60)
display(Image(filename=str(report.with_suffix(".png"))))
pod.upload(api, HF_REPO_ID, folders=[FOLDER], reports=[f"{EXPERIMENT}_*.md", f"{EXPERIMENT}_*.png"])